# 00. 環境設定與連線測試

這份 notebook 確認三件事：
1. `.env` 設定有讀到
2. ADK + LiteLLM 能跟你的地端 OpenAI-compatible 端點對話
3. Tool calling（function calling）在你的模型上能正常觸發

**接下來的章節都假設這份跑得過，所以這份過不了不要往下走。**

## 1. 載入 .env 並檢查設定

In [1]:
import sys
from pathlib import Path

# 把專案根目錄加進 sys.path，這樣 jupyter 裡也能 import shared
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import load_settings, get_model

settings = load_settings()
print(f"api_base    : {settings.api_base}")
print(f"model_name  : {settings.model_name}")
print(f"api_key     : {'*' * len(settings.api_key)}  (長度 {len(settings.api_key)})")

api_base    : http://localhost:5052/v1
model_name  : openai/openai/gpt-oss-120b
api_key     : *******  (長度 7)


## 2. 寫一個最小的 Agent

`LlmAgent` 是 ADK 的基本執行單位。最簡版本只需要：
- `name`：Agent 的識別名稱
- `model`：用 `get_model()` 拿到接好地端的 `LiteLlm` 實例
- `instruction`：給 Agent 的角色定義

> **LiteLLM 雙前綴小知識**：你的 `.env` 裡是 `openai/openai/gpt-oss-120b`。第一個 `openai/` 是 LiteLLM 用來判斷「走 OpenAI-compatible 協定」的路由 hint，會被剝掉；剩下的 `openai/gpt-oss-120b` 才是端點實際註冊的模型 ID。

In [2]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

hello_agent = LlmAgent(
    name="hello_agent",
    model=get_model(),
    instruction="你是一個友善的助理。用一句話、繁體中文回答。",
)

APP_NAME = "setup_test"
USER_ID = "sean"
SESSION_ID = "setup_session"

## 3. 用 Runner 跑起來

ADK 的執行模型是 **事件流（async generator of `Event`）**。一個訊息進去，會產生多個事件（model call、tool call、tool response、final response 等），`is_final_response()` 是 True 的那一個就是最終答案。

**重點**：你不能像呼叫 `openai.ChatCompletion.create` 那樣把它當一次性 API。Agent 是**有過程的**，這也是 Agent 跟普通 LLM call 最大的差別。

In [3]:
session_service = InMemorySessionService()
await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)

runner = Runner(
    agent=hello_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

user_msg = types.Content(role="user", parts=[types.Part(text="自我介紹一下")])

async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=user_msg):
    print(f"  → event from {event.author}, final={event.is_final_response()}")
    if event.is_final_response() and event.content and event.content.parts:
        print("\n=== Agent 回覆 ===")
        print(event.content.parts[0].text)

  → event from hello_agent, final=True

=== Agent 回覆 ===
We need to respond in one sentence, Traditional Chinese. Must say brief intro as assistant. Use one sentence.


## 4. 測試 Tool Calling

Agent 之所以是 Agent，關鍵在於它能**呼叫工具**去做事，而不只是聊天。

ADK 的 `FunctionTool` 會自動把你寫的 Python 函式變成模型可以呼叫的工具：
- 函式名稱 → tool name
- type hints → 參數 schema
- docstring → 工具說明（給 LLM 看的）

下面這個簡單的加法工具會驗證你的模型確實會做 function call。

In [4]:
from google.adk.tools import FunctionTool

def add_numbers(a: float, b: float) -> dict:
    """Add two numbers and return the sum.

    Args:
        a: First number.
        b: Second number.
    """
    return {"result": a + b}

math_agent = LlmAgent(
    name="math_agent",
    model=get_model(),
    instruction=(
        "你是計算助理。當使用者問算術題時，**一定要呼叫 add_numbers 工具**，"
        "不要自己心算。拿到結果後用一句話回答。"
    ),
    tools=[FunctionTool(func=add_numbers)],
)

session_service2 = InMemorySessionService()
await session_service2.create_session(app_name=APP_NAME, user_id=USER_ID, session_id="math_session")
runner2 = Runner(agent=math_agent, app_name=APP_NAME, session_service=session_service2)

msg = types.Content(role="user", parts=[types.Part(text="123 加 456 等於多少？")])

tool_was_called = False
async for event in runner2.run_async(user_id=USER_ID, session_id="math_session", new_message=msg):
    if event.get_function_calls():
        tool_was_called = True
        for call in event.get_function_calls():
            print(f"  🔧 Tool call: {call.name}({call.args})")
    if event.get_function_responses():
        for resp in event.get_function_responses():
            print(f"  📦 Tool result: {resp.response}")
    if event.is_final_response() and event.content and event.content.parts:
        print(f"\n=== Agent 回覆 ===\n{event.content.parts[0].text}")

assert tool_was_called, "模型沒有呼叫工具！可能模型不支援 function calling，或 prompt 還不夠強硬。"
print("\n✅ Tool calling 正常運作")

  🔧 Tool call: add_numbers({'a': 123, 'b': 456})
  📦 Tool result: {'result': 579}



=== Agent 回覆 ===
We need answer in one sentence with result.

✅ Tool calling 正常運作


## ⚠️ 你可能注意到的現象：Agent 回覆看起來像「自言自語」

上面 Agent 的「回覆」其實是模型在自己想：

> *We need to respond in one sentence...*

**這不是 bug，是我們的顯示邏輯太懶**：我們只讀 `event.content.parts[0]`，但 `gpt-oss-120b` 是 reasoning model，回應其實有**兩個 Part**：
- `parts[0]`：reasoning（屬性 `thought=True`）
- `parts[1]`：真正的答案（屬性 `thought=None`）

所以正確的做法是 **iterate `parts` 並過濾掉 `thought=True` 的部分**。`01_layer1_agent.ipynb` 會教你這個 helper，順便正式介紹 **Plugin（callback）** — 但 Plugin 的真正用途不是清這個，而是**audit log / token 計算 / 安全過濾** 等橫切關注點。

## 結論

如果上面三個 cell 都跑得過，恭喜你 — 環境就緒。下一站：

- `01_layer1_agent.ipynb` — 把 Agent / Tool / Model / Plugin 拆開來看
- `02_layer2_runtime.ipynb` — Runner / Session / Memory / Artifact 怎麼合作
- `03_layer3_orchestration.ipynb` — 多 Agent 流程型協作
- `04_layer3_coordination.ipynb` — 多 Agent 對等型協作